# 6. Воспроизводимый отчёт

**Цель:** выполнить поддерживаемый `GraphEVTPipeline` и сохранить численный результат, manifest и краткий отчёт. Ни LLM, ни вымышленные поля результата для этого не требуются.

In [1]:
from pathlib import Path
from importlib.metadata import version
from hashlib import sha256
import json
import numpy as np
import pandas as pd
from graph_evt_agent import EVTConfig, GraphConfig, GraphEVTPipeline, InputConfig

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/generated/additive_fractal_series.csv").exists())
DATA = ROOT / "data/generated/additive_fractal_series.csv"
SEED = 42
df = pd.read_csv(DATA)
x = df["value"].to_numpy()
event_mask = df["is_extreme"].to_numpy(dtype=bool)
assert np.isfinite(x).all() and x.ndim == 1
assert event_mask.shape == x.shape and event_mask.any()
event_start = int(np.flatnonzero(event_mask)[0])
event_end = int(np.flatnonzero(event_mask)[-1])

In [2]:
BASELINE_SIZE = event_start
parameters = {
    "baseline_size": BASELINE_SIZE,
    "tail_quantile": 0.95,
    "alarm_probability": 0.999,
    "persistence": 3,
}
evt = EVTConfig(**parameters)
result = GraphEVTPipeline(
    evt,
    graph=GraphConfig(random_state=SEED),
    input_config=InputConfig(missing="error"),
    verbose=True,
).run_detailed(x)
detection = result.detection
alarm_index = detection.time_index
detected_in_labeled_event = alarm_index is not None and bool(event_mask[int(alarm_index)])
detection_delay = int(alarm_index) - event_start if detected_in_labeled_event else None

artifact_dir = ROOT / "results/1-d/06_report"
artifact_dir.mkdir(parents=True, exist_ok=True)
indicator_path = artifact_dir / "indicator.csv"
pd.DataFrame({
    "time": df["timestamp"],
    "value": x,
    "is_extreme": event_mask,
    "indicator": detection.indicator,
    "above_alarm_threshold": detection.indicator > detection.alarm_threshold,
    "is_baseline": np.arange(len(x)) < BASELINE_SIZE,
}).to_csv(indicator_path, index=False)

Graph-EVT pipeline:   0%|          | 0/3 [00:00<?, ?it/s]

In [3]:
manifest = {
    "data_file": str(DATA.relative_to(ROOT)),
    "data_sha256": sha256(DATA.read_bytes()).hexdigest(),
    "library": "graph-evt-agent",
    "library_version": version("graph-evt-agent"),
    "seed": SEED,
    "parameters": parameters,
    "baseline_selection": "labeled_event_start_oracle_assisted",
    "labeled_event": {"start": event_start, "end": event_end},
    "input_profile": {
        "kind": result.input_profile.kind,
        "n_time": result.input_profile.n_time,
        "n_channels": result.input_profile.n_channels,
    },
    "detection": {
        "detected": detection.detected,
        "time_index": alarm_index,
        "detected_in_labeled_event": detected_in_labeled_event,
        "detection_delay_samples": detection_delay,
        "alarm_threshold": float(detection.alarm_threshold),
    },
    "artifacts": {"indicator_csv": indicator_path.name},
    "limitations": [
        "synthetic injected event; its label is used only for baseline selection and evaluation",
        "oracle-assisted baseline selection is not available for unlabeled real data",
        "two-sided deviations are combined in one indicator",
        "no graph or source ranking is available for one channel",
        "results depend on baseline and EVT configuration",
    ],
}
with (artifact_dir / "manifest.json").open("w", encoding="utf-8") as file:
    json.dump(manifest, file, ensure_ascii=False, indent=2)

status = "обнаружено" if detection.detected else "не обнаружено"
event_status = "да" if detected_in_labeled_event else "нет"
delay_text = f"{detection_delay} отсчётов" if detection_delay is not None else "не определена"
summary = f"""# Краткий отчёт

На {len(x)} наблюдениях устойчивое превышение {status}.
Alarm-порог: {detection.alarm_threshold:.6g}; первый индекс: {alarm_index}.
Размеченный интервал события: [{event_start}, {event_end}]; срабатывание внутри события: {event_status}.
Задержка относительно начала события: {delay_text}.
Численный ряд indicator сохранён в `indicator.csv`, параметры и ограничения — в `manifest.json`.
"""
(artifact_dir / "report.md").write_text(summary, encoding="utf-8")
manifest

{'data_file': 'data/generated/additive_fractal_series.csv',
 'data_sha256': 'cf98a3c41aa23efc5d6e6461f5c56ef1280111278e6ebab37e6cd375b4bb0af9',
 'library': 'graph-evt-agent',
 'library_version': '0.1.0',
 'seed': 42,
 'parameters': {'baseline_size': 1000,
  'tail_quantile': 0.95,
  'alarm_probability': 0.999,
  'persistence': 3},
 'baseline_selection': 'labeled_event_start_oracle_assisted',
 'labeled_event': {'start': 1000, 'end': 1149},
 'input_profile': {'kind': 'univariate', 'n_time': 1500, 'n_channels': 1},
 'detection': {'detected': True,
  'time_index': 1124,
  'detected_in_labeled_event': True,
  'detection_delay_samples': 124,
  'alarm_threshold': 2.066649951679863},
 'artifacts': {'indicator_csv': 'indicator.csv'},
 'limitations': ['synthetic injected event; its label is used only for baseline selection and evaluation',
  'oracle-assisted baseline selection is not available for unlabeled real data',
  'two-sided deviations are combined in one indicator',
  'no graph or source 

In [4]:
required = {"data_sha256", "library_version", "parameters", "input_profile", "detection", "artifacts", "limitations"}
missing = required - set(manifest)
assert not missing, f"Неполный manifest: {sorted(missing)}"
assert len(manifest["data_sha256"]) == 64
summary

'# Краткий отчёт\n\nНа 1500 наблюдениях устойчивое превышение обнаружено.\nAlarm-порог: 2.06665; первый индекс: 1124.\nРазмеченный интервал события: [1000, 1149]; срабатывание внутри события: да.\nЗадержка относительно начала события: 124 отсчётов.\nЧисленный ряд indicator сохранён в `indicator.csv`, параметры и ограничения — в `manifest.json`.\n'

Manifest связывает численный результат с точными байтами данных, версией библиотеки и параметрами `GraphEVTPipeline`. Метка `is_extreme` используется только для выбора чистого baseline до события и последующей оценки, но не передаётся детектору. Это oracle-assisted проверка на синтетическом ряду; для реальных неразмеченных данных baseline необходимо задать по отдельному протоколу.

**Самопроверка:** зачем хранить hash данных? Как загрязнение baseline событием меняет вывод? Какие ограничения остаются у одномерной детекции?